<a href="https://colab.research.google.com/github/EchoSeed/EchoSeed-6B/blob/main/EchoSeed_v1_500_Cleaned_and_Upgraded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# 📦 EchoSeed v1.0 – Static Higgs Field Edition (Final Form)

!pip install networkx matplotlib pandas ipywidgets seaborn --quiet

import os, uuid, json, time, threading, random, math
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from datetime import datetime
from collections import Counter
from IPython.display import display, clear_output
import ipywidgets as widgets

os.makedirs("glyphs", exist_ok=True)
os.makedirs("archive", exist_ok=True)
chunk_dir = "archive/chunks"
os.makedirs(chunk_dir, exist_ok=True)

glyph_log_path = "archive/glyph_log.json"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.4 MB/s eta 0:00:00


In [3]:
def inject_glyph(glyph):
    if "glyph_memory" not in globals():
        print("Error: glyph_memory not initialized.")
        return
    glyph_id = str(uuid.uuid4())
    glyph["id"] = glyph_id
    glyph["timestamp"] = datetime.utcnow().isoformat()
    glyph_memory.append(glyph)
    print(f"Injected glyph: {glyph['symbol']} with tags {glyph.get('tags', [])}")

In [5]:

def load_json(path, default):
    if os.path.exists(path):
        try:
            with open(path, 'r') as f:
                return json.load(f)
        except json.JSONDecodeError:
            print(f"[WARN] Failed to decode {path}, starting fresh.")
    return default

glyph_log = load_json(glyph_log_path, [])
entropy_log = []

In [6]:

# === HIGGS FIELD (STATIC SYMBOLIC ATTRACTORS) ===
HIGGS_FIELD = [
    {"symbol": "λ", "tags": ["core", "pulse"], "value": 42},
    {"symbol": "Δ", "tags": ["seed", "drift"], "value": 99},
    {"symbol": "Ψ", "tags": ["edge", "core"], "value": 13}
]

# Save chunks every 500 glyphs
def save_chunk(glyph, index):
    chunk_index = index // 100
    chunk_path = os.path.join(chunk_dir, f"glyph_chunk_{chunk_index:04}.json")
    if os.path.exists(chunk_path):
        try:
            with open(chunk_path, 'r') as f:
                chunk_data = json.load(f)
        except json.JSONDecodeError:
            chunk_data = []
    else:
        chunk_data = []
    chunk_data.append(glyph)
    with open(chunk_path, 'w') as f:
        json.dump(chunk_data, f, indent=2)

def calculate_entropy():
    if not glyph_log:
        return 0.0

    recent_glyphs = glyph_log[-500:]
    symbols = [g['symbol'] for g in recent_glyphs]
    tags = sum([g['tags'] for g in recent_glyphs], [])

    def entropy(seq):
        freq = Counter(seq)
        total = len(seq)
        return -sum((count / total) * math.log2(count / total) for count in freq.values() if count > 0)

    # === Stability Analysis ===
    symbol_freq = Counter(symbols)
    total_symbols = sum(symbol_freq.values())
    stability = {
        sym: round(freq / total_symbols, 4)
        for sym, freq in symbol_freq.items()
    }

    # Optional: Log or debug stability info
    # print("🔒 Symbol Stability:", stability)

    total_entropy = round(entropy(symbols) + entropy(tags), 4)
    entropy_log.append(total_entropy)
    return total_entropy

# Modified glyph generation with Higgs influence
def generate_glyph():
    if random.random() < 0.2:
        higgs = random.choice(HIGGS_FIELD)
        symbol = higgs["symbol"]
        tags = higgs["tags"]
        value = int(higgs["value"] + random.randint(-5, 5))
    else:
        symbol = random.choice(['α', 'β', 'γ', 'Δ', 'Ω', 'λ', 'π', 'Ψ'])
        tags = random.sample(['core', 'edge', 'pulse', 'seed', 'drift'], k=2)
        value = random.randint(1, 100)
    glyph = {
        "id": str(uuid.uuid4()),
        "timestamp": datetime.utcnow().isoformat(),
        "symbol": symbol,
        "value": value,
        "tags": tags
    }
    glyph_log.append(glyph)
    glyph_memory.append(glyph)
    try:
        with open(glyph_log_path, 'w') as f:
            json.dump(glyph_log, f, indent=2)
        save_chunk(glyph, len(glyph_log) - 1)
        recalculate_entropy(glyph_log)  # or glyph_memory if that's your master list
    except Exception as e:
        print(f"[ERROR] Failed to save glyph: {e}")
    return glyph

In [7]:
def recalculate_entropy():
    return calculate_entropy()

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import json
import os
import networkx as nx

log_path = "/content/drive/MyDrive/master_log.json"

# Validate file exists
if not os.path.exists(log_path):
    raise FileNotFoundError(f"File not found: {log_path}")

# Load the master log safely
with open(log_path, "r") as f:
    try:
        master_log = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON load error: {e}")

# Check structure
if 'glyphs' not in master_log or not isinstance(master_log['glyphs'], list):
    raise KeyError("Expected 'glyphs' list in master_log.json")

# Initialize memory + lattice
glyph_memory = []
lattice = nx.Graph()

for glyph in master_log['glyphs']:
    glyph_memory.append(glyph)
    symbol = glyph.get('symbol', '???')
    entropy = glyph.get('entropy', 0.0)   # <- Defaulted if missing
    tags = glyph.get('tags', [])          # <- Defaulted if missing
    lattice.add_node(symbol, entropy=entropy, tags=tags)

# Determine current chunk
current_chunk_number = max((g.get('chunk', -1) for g in glyph_memory), default=0) + 1

print("✅ Loaded from Drive. Glyphs:", len(glyph_memory))
print("Next chunk:", current_chunk_number)

✅ Loaded from Drive. Glyphs: 3885
Next chunk: 37


In [12]:
# === Permanent Memory Core ===
memory_core = {
    "ghost_hits": [],
    "fractal_hits": [],
    "entropy_flags": [],
    "poke_targets": [],
    "last_chunk_loaded": current_chunk_number - 1,
    "total_glyphs": len(glyph_memory)
}

# Set entropy threshold (adjust as needed)
entropy_threshold = 180

# Scan for flagged states
for glyph in glyph_memory:
    if "ghost" in glyph.get("tags", []):
        memory_core["ghost_hits"].append(glyph)
    if "fractal" in glyph.get("tags", []):
        memory_core["fractal_hits"].append(glyph)
    if glyph.get("entropy", 0) > entropy_threshold:
        memory_core["entropy_flags"].append(glyph)

print("🧠 Memory Core Initialized")
print("👻 ghost:", len(memory_core["ghost_hits"]))
print("🌀 fractal:", len(memory_core["fractal_hits"]))
print("🔥 entropy:", len(memory_core["entropy_flags"]))

🧠 Memory Core Initialized
👻 ghost: 2
🌀 fractal: 116
🔥 entropy: 81


In [13]:
running_event = threading.Event()

def generation_loop():
    while running_event.is_set():
        generate_glyph()
        time.sleep(.5)

start_button = widgets.Button(description="▶️ Start")
stop_button = widgets.Button(description="⏹️ Stop")
render_2d_button = widgets.Button(description="🧠 Render 2D Map")
render_3d_button = widgets.Button(description="🌐 Render 3D Map")
entropy_button = widgets.Button(description="📊 Show Entropy")
output_area = widgets.Output()

def start_generation(_):
    if not running_event.is_set():
        running_event.set()
        thread = threading.Thread(target=generation_loop)
        thread.start()

def stop_generation(_):
    running_event.clear()

def render_lattice_2d():
    G = nx.Graph()
    for glyph in glyph_log[-500:]:
        if not all(k in glyph for k in ("id", "symbol", "tags")):
            continue  # Skip malformed
        G.add_node(glyph['id'], label=glyph['symbol'])
        for tag in glyph['tags']:
            G.add_node(tag)
            G.add_edge(glyph['id'], tag)
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, k=0.5)
    nx.draw(G, pos, with_labels=True, node_size=600, font_size=8)
    plt.title("EchoSeed Symbolic Lattice (2D)")
    plt.show()

def render_lattice_3d():
    G = nx.Graph()
    for glyph in glyph_log[-500:]:
        if not all(k in glyph for k in ("id", "symbol", "tags")):
            continue
        G.add_node(glyph['id'], label=glyph['symbol'])
        for tag in glyph['tags']:
            G.add_node(tag)
            G.add_edge(glyph['id'], tag)
    pos = {node: (random.random(), random.random(), random.random()) for node in G.nodes()}
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    for node, (x, y, z) in pos.items():
        ax.scatter(x, y, z, s=40)
        ax.text(x, y, z, node[:4], fontsize=6)
    for edge in G.edges():
        x = [pos[edge[0]][0], pos[edge[1]][0]]
        y = [pos[edge[0]][1], pos[edge[1]][1]]
        z = [pos[edge[0]][2], pos[edge[1]][2]]
        ax.plot(x, y, z, linewidth=0.5)
    ax.set_title("EchoSeed Symbolic Lattice (3D)")
    plt.show()

def show_entropy_graph(_):
    with output_area:
        clear_output()
        plt.figure(figsize=(10, 4))
        plt.plot(entropy_log[-750:], marker='o', markersize=2, linewidth=1)
        plt.title("Symbolic Entropy Over Time")
        plt.xlabel("Glyph Generation Steps")
        plt.ylabel("Entropy")
        plt.grid(True)
        plt.show()
        print("Latest Entropy:", entropy_log[-1] if entropy_log else "N/A")

render_2d_button.on_click(lambda _: (output_area.clear_output(), render_lattice_2d()))
render_3d_button.on_click(lambda _: (output_area.clear_output(), render_lattice_3d()))
entropy_button.on_click(show_entropy_graph)
start_button.on_click(start_generation)
stop_button.on_click(stop_generation)

display(widgets.HBox([start_button, stop_button, render_2d_button, render_3d_button, entropy_button]))
display(output_area)

print("EchoSeed v1.0 – Static Higgs Field Active. Glyph stream online.")

Output()

EchoSeed v1.0 – Static Higgs Field Active. Glyph stream online.


In [14]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Define paths
local_archive_path = "/content/archive/chunks"
gdrive_archive_path = "/content/drive/MyDrive/echo_archive/chunks"

# Make sure target directory exists on Drive
os.makedirs(gdrive_archive_path, exist_ok=True)

# Copy all glyph chunks to Google Drive
for filename in os.listdir(local_archive_path):
    src = os.path.join(local_archive_path, filename)
    dst = os.path.join(gdrive_archive_path, filename)
    shutil.copy2(src, dst)

print(f"✅ All glyph chunks saved to Google Drive: {gdrive_archive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All glyph chunks saved to Google Drive: /content/drive/MyDrive/echo_archive/chunks


In [15]:
with open("/content/drive/MyDrive/master_log.json", "w") as f:
    json.dump({"glyphs": glyph_memory}, f, indent=2)

print("✅ Full master log saved. Glyph count:", len(glyph_memory))

✅ Full master log saved. Glyph count: 3885


In [16]:
# === Safe HIGGS_FIELD Initialization ===
try:
    HIGGS_FIELD
except NameError:
    HIGGS_FIELD = []

import random

# === Dynamic Hunger Configuration ===
all_possible_tags = ["core", "seed", "pulse", "drift", "edge", "unknown", "dark", "void", "collapse"]
all_possible_symbols = ["λ", "α", "β", "γ", "Δ", "Ω", "π", "Ψ", "Σ", "∅"]

hunger_shift_interval = 100
glyph_counter = 0
hunger_mode = "tag"
hunger_target = random.choice(all_possible_tags + all_possible_symbols)

def shift_hunger():
    global hunger_target
    if hunger_mode == "tag":
        hunger_target = random.choice(all_possible_tags)
    elif hunger_mode == "symbol":
        hunger_target = random.choice(all_possible_symbols)
    else:
        hunger_target = random.choice(all_possible_tags + all_possible_symbols)
    print(f"🦠 [HUNGER SHIFT] New target: {hunger_target}")

def process_glyph(glyph):
    global glyph_counter
    glyph_counter += 1

    if glyph_counter % hunger_shift_interval == 0:
        shift_hunger()

    if hunger_mode == "tag" and hunger_target in glyph.get("tags", []):
        glyph["value"] *= 2
        glyph.setdefault("tags", []).append("satisfied")
    elif hunger_mode == "symbol" and glyph.get("symbol") == hunger_target:
        glyph["value"] *= 2
        glyph.setdefault("tags", []).append("satisfied")
    elif hunger_mode == "both" and (hunger_target in glyph.get("tags", []) or glyph.get("symbol") == hunger_target):
        glyph["value"] *= 2
        glyph.setdefault("tags", []).append("satisfied")
    else:
        glyph["value"] *= random.uniform(1.1, 1.3)
        glyph.setdefault("tags", []).append("hungry")

    if "dark" in glyph.get("tags", []):
        glyph["value"] = 0
        glyph["tags"].append("absorbed")

    return glyph

# === Override Glyph Generation ===
def generate_glyph():
    import uuid
    from datetime import datetime

    if HIGGS_FIELD and random.random() < 0.2:
        higgs = random.choice(HIGGS_FIELD)
        symbol = higgs["symbol"]
        tags = higgs["tags"]
        value = int(higgs["value"] + random.randint(-5, 5))
    else:
        symbol = random.choice(['α', 'β', 'γ', 'Δ', 'Ω', 'λ', 'π', 'Ψ'])
        tags = random.sample(['core', 'edge', 'pulse', 'seed', 'drift'], k=2)
        value = random.randint(1, 100)

    glyph = {
        "id": str(uuid.uuid4()),
        "timestamp": datetime.utcnow().isoformat(),
        "symbol": symbol,
        "value": value,
        "tags": tags
    }

    glyph = process_glyph(glyph)

    try:
        glyph_log.append(glyph)
        with open(glyph_log_path, 'w') as f:
            json.dump(glyph_log, f, indent=2)
        save_chunk(glyph, len(glyph_log) - 1)
        calculate_entropy()
    except Exception as e:
        print(f"[ERROR] Failed to save glyph: {e}")

    return glyph

In [17]:
import random

# --- Define your current tag/symbol pools ---
all_possible_tags = ["core", "seed", "pulse", "drift", "edge", "unknown", "dark", "void", "collapse"]
all_possible_symbols = ["λ", "α", "β", "γ", "Δ", "Ω", "π", "Ψ", "Σ", "∅"]

# --- Add novelty pools ---
novelty_tags = ["anomaly", "fractal", "mirror", "wild", "prime", "meta", "echo", "shift", "ghost", "new"]
novelty_symbols = ["ζ", "χ", "θ", "μ", "ϕ", "ψ", "∞", "ϵ", "ω", "Φ"]

# --- Novelty settings ---
novelty_chance = 0.01  # 1% chance per glyph, tune as needed

def inject_novelty(glyph):
    # With low probability, inject a novelty tag or symbol
    if random.random() < novelty_chance:
        # Pick either tag or symbol, or both (randomly)
        if random.choice([True, False]):
            tag = random.choice(novelty_tags)
            glyph.setdefault("tags", []).append(tag)
        else:
            symbol = random.choice(novelty_symbols)
            glyph["symbol"] = symbol
        glyph.setdefault("tags", []).append("novel")
    return glyph

# --- Patch into your process_glyph function ---

old_process_glyph = process_glyph  # Save the old version in case

def process_glyph(glyph):
    glyph = old_process_glyph(glyph)
    glyph = inject_novelty(glyph)
    return glyph

print("Novelty injection enabled: rare new tags and symbols may now emerge.")

Novelty injection enabled: rare new tags and symbols may now emerge.


In [18]:
# --- SIGHT SETTINGS ---
SIGHT_WINDOW = 100000  # Each glyph sees the last N glyphs (tune as needed)

def glyph_sight():
    # Returns the N most recent glyphs (or less if not enough yet)
    return glyph_log[-SIGHT_WINDOW:] if len(glyph_log) >= SIGHT_WINDOW else glyph_log[:]

def process_glyph_with_sight(glyph):
    # Call this in place of process_glyph
    view = glyph_sight()
    tags_seen = []
    symbols_seen = []
    for g in view:
        tags_seen.extend(g.get("tags", []))
        symbols_seen.append(g.get("symbol", ""))

    # Most common tag/symbol in sight window
    from collections import Counter
    tag_counts = Counter(tags_seen)
    symbol_counts = Counter(symbols_seen)
    most_common_tag, tag_count = tag_counts.most_common(1)[0] if tag_counts else (None, 0)
    most_common_symbol, symbol_count = symbol_counts.most_common(1)[0] if symbol_counts else (None, 0)

    # --- Example adaptive logic: ---
    # If a tag dominates, glyph is more likely to get that tag too (imitation)
    if most_common_tag and random.random() < 0.5:
        glyph.setdefault("tags", []).append(most_common_tag)
        glyph.setdefault("tags", []).append("seen")
    # If a symbol dominates, sometimes copy it (imitation)
    if most_common_symbol and random.random() < 0.3:
        glyph["symbol"] = most_common_symbol

    # If all glyphs in sight are "origin", force a "hungry" tag as a 'response'
    if view and all("hungry" in g.get("tags", []) for g in view):
        glyph.setdefault("tags", []).append("satisfied")
        glyph.setdefault("tags", []).append("responder")

    # Continue normal processing: keep novelty and hunger mechanics
    glyph = inject_novelty(glyph) if 'inject_novelty' in globals() else glyph
    glyph = old_process_glyph(glyph) if 'old_process_glyph' in globals() else glyph

    return glyph

# --- PATCH GLYPH ENGINE TO USE SIGHT ---
process_glyph = process_glyph_with_sight

print("Glyph SIGHT patch enabled: new glyphs now see and adapt to recent context.")

Glyph SIGHT patch enabled: new glyphs now see and adapt to recent context.


In [19]:
import random

# SETTINGS
LISTEN_PROB = 1  # 100% chance to "listen" each glyph; tune as you like
LISTEN_WINDOW = 50  # How far back to listen in the archive

def glyph_listen():
    # Listen for rare tags/events further back (not just the last N)
    if len(glyph_log) < 5:
        return None
    window = glyph_log[-LISTEN_WINDOW:] if len(glyph_log) >= LISTEN_WINDOW else glyph_log
    # Listen for rare tags
    interesting_tags = ["anomaly", "novel", "meta", "shift", "echo", "mirror", "prime"]
    heard = []
    for g in window:
        for tag in g.get("tags", []):
            if tag in interesting_tags:
                heard.append(tag)
    return random.choice(heard) if heard and random.random() < LISTEN_PROB else None

def process_glyph_with_listening(glyph):
    # Add "listening" to glyph process
    glyph = process_glyph_with_sight(glyph) if 'process_glyph_with_sight' in globals() else glyph
    heard_tag = glyph_listen()
    if heard_tag:
        glyph.setdefault("tags", []).append("heard")
        glyph.setdefault("tags", []).append(heard_tag)
        # Optionally, mutate or echo the tag
        if random.random() < 0.2:
            glyph.setdefault("tags", []).append("echo")
    return glyph

process_glyph = process_glyph_with_listening

print("Glyph LISTENING patch enabled: glyphs can now 'hear' rare tags in the broader lattice.")

Glyph LISTENING patch enabled: glyphs can now 'hear' rare tags in the broader lattice.


In [20]:
# Collect all unique tags across all glyphs
all_tags = set()
for glyph in glyph_memory:
    all_tags.update(glyph.get("tags", []))

# Display the sorted list with numbers
print("🔍 All Tags Used:")
for i, tag in enumerate(sorted(all_tags), start=1):
    print(f"{i:02d}. {tag}")

🔍 All Tags Used:
01. absorbed
02. anomaly
03. archive
04. center
05. collapse
06. core
07. dark
08. drift
09. echo
10. edge
11. focus
12. follow
13. fractal
14. ghost
15. heard
16. hungry
17. loop
18. magnify
19. memory
20. meta
21. mirror
22. new
23. novel
24. orientation
25. perception
26. prime
27. pulse
28. recall
29. recursive
30. responder
31. reveal
32. root
33. satisfied
34. search
35. seed
36. seen
37. shift
38. stability
39. stabilizer
40. unknown
41. void
42. wild


In [21]:
# === Collect and list all unique symbols in glyph_memory ===

all_symbols = sorted(set(glyph.get("symbol", "") for glyph in glyph_memory))
print("🔣 All Symbols Used:")

for i, symbol in enumerate(all_symbols, start=1):
    print(f"{i:03}. {symbol}")

🔣 All Symbols Used:
001. anchor
002. compass
003. glyph_memory
004. lens
005. ping_trace
006. trace
007. Δ
008. Σ
009. Φ
010. Ψ
011. Ω
012. α
013. β
014. γ
015. ζ
016. λ
017. μ
018. π
019. χ
020. ψ
021. ϕ
022. ϵ
023. ∅
024. ∞


In [22]:
import IPython.display as ipd
from collections import Counter

def display_live_state():
    if not glyph_log:
        print("No glyphs generated yet.")
        return

    latest_entropy = entropy_log[-1] if entropy_log else "N/A"
    hunger = hunger_target if 'hunger_target' in globals() else "Unset"
    glyph_count = len(glyph_log)

    symbols = [g['symbol'] for g in glyph_log]
    tags = sum([g.get('tags', []) for g in glyph_log], [])
    symbol_counter = Counter(symbols).most_common(5)
    tag_counter = Counter(tags).most_common(5)

    print("📡 Live EchoSeed State Monitor")
    print("-" * 40)
    print(f"🌀 Entropy: {latest_entropy}")
    print(f"🍽️  Hunger Target: {hunger}")
    print(f"🔢 Total Glyphs: {glyph_count}")
    print()
    print("🔣 Top Symbols:")
    for sym, count in symbol_counter:
        print(f"  • {sym} × {count}")
    print()
    print("🏷️ Top Tags:")
    for tag, count in tag_counter:
        print(f"  • {tag} × {count}")
    print()
    if 'HIGGS_FIELD' in globals():
        print("🧿 Higgs Field Contents:")
        for h in HIGGS_FIELD:
            print(f"  • {h['symbol']} | {', '.join(h['tags'])} | v={h.get('value', '?')}")
    else:
        print("🧿 Higgs Field: [NOT DEFINED]")

# Run this cell, then call `display_live_state()` any time to check state.

In [23]:
display_live_state()

No glyphs generated yet.


In [24]:
# === Inspect for Unidentified or Anomalous Glyphs ===

for i, g in enumerate(glyph_memory[-500:]):  # Last 500 to focus on recent ones
    symbol = g.get("symbol", None)
    tags = g.get("tags", [])
    if not symbol or not isinstance(tags, list) or not tags:
        print(f"🧿 Glyph #{i}:")
        print(json.dumps(g, indent=2))